In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import ipaddress
import json
import logging
from pathlib import Path
import socket

import requests
import pandas as pd

In [2]:
from constants import (
    STARTDUST_URL,
    STARTDUST_ELASTIC_INDEX,
    STARDUST_QUERY_RANGE,
    EUROPEAN_COUNTRIES,
    EUROPEAN_MESSAGES_FILENAME_PREFIX,
)

In [3]:
PROGRESS_MESSAGE_FREQUENCY = 10000

def load_european_fireflies(
    time_range=STARDUST_QUERY_RANGE,
    countries=EUROPEAN_COUNTRIES,
    page_size=1000,
):
    num_records = 0

    # Open a Point in Time query
    rsp = requests.post(
        f'{STARTDUST_URL}/{STARTDUST_ELASTIC_INDEX}/_pit',
        params={'keep_alive': '2m'},
        timeout=60,
    )
    rsp.raise_for_status()

    pit_id = rsp.json()["id"]

    try:
        search_after = None

        while True:

            query = {
                "size": page_size,
                "pit": {
                    "id": pit_id,
                    "keep_alive": "2m",
                },
                "query": {
                    "bool": {
                        "filter": [
                            {
                                "range": {
                                    "@timestamp": {
                                        "gte": f"now-{time_range}",
                                        "lte": "now",
                                    }
                                }
                            },
                            {
                                "bool": {
                                    "should": [
                                        {
                                            "terms": {"meta.cric.src.country.keyword": list(countries)}
                                        },
                                        {
                                            "terms": {"meta.cric.dst.country.keyword": list(countries)}
                                        },
                                    ],
                                    "minimum_should_match": 1,
                                }
                            },
                        ]
                    }
                },
                "sort": [{"_shard_doc": "asc"}]
            }

            if search_after is not None:
                query["search_after"] = search_after

            rsp = requests.post(
                f"{STARTDUST_URL}/_search",
                json=query,
                timeout=60,
            )
            rsp.raise_for_status()

            response = rsp.json()

            # Elasticsearch may return an updated PIT ID.
            pit_id = response.get("pit_id", pit_id)

            hits = response["hits"]["hits"]

            if not hits:
                break

            for hit in hits:
                num_records += 1
                if num_records % PROGRESS_MESSAGE_FREQUENCY == 0:
                    logging.warning(f'{num_records} hits')
                yield hit["_source"]

            search_after = hits[-1]["sort"]

            if len(hits) < page_size:
                break

    finally:
        # Always close the PIT, including when iteration stops early.
        requests.delete(
            f"{STARTDUST_URL}/_pit",
            json={"id": pit_id},
            timeout=30,
        )

In [4]:
# MESSAGES_PER_FILE = 1000000
MESSAGES_PER_FILE = 20000

# european_messages = load_european_fireflies()
european_messages = load_european_fireflies(time_range='4m')

batch = []
file_number = 1
total_message_count = 0

def _save_batch():
    filename = f"{EUROPEAN_MESSAGES_FILENAME_PREFIX}-{file_number:03d}.json"
    with open(filename, 'w') as f:
        json.dump(batch, f)
    logging.warning(f'saved {len(batch)} messages to {filename}')
    
for _f in european_messages:
    batch.append(_f)
    if len(batch) >= MESSAGES_PER_FILE:
        _save_batch()
        total_message_count += len(batch)
        file_number += 1
        batch = []

if batch:
    _save_batch()
    total_message_count += len(batch)

logging.warning(f'total number of messages save: {total_message_count}')